In [2]:
# === Setup en verbinding ===
import ConnectionConfig as cc
from pyspark.sql.functions import (
    col, year, month, dayofmonth, weekofyear, date_format,
    weekday, when, expr, to_date, row_number
)
from pyspark.sql.window import Window

debugging_mode = True


In [3]:
#  Extract
cc.setupEnvironment()
cc.setupEnvironment()
print(cc.config.sections())

spark = cc.startLocalCluster("DIM_DATE", 4)
spark.getActiveSession()
cc.config.read('config.ini')
cc.set_connectionProfile("catchem")


Environment variables are set...
Environment variables are set...
['default', 'tutorial_op', 'catchem', 'kafka']


In [7]:
#EXTRACT
date_src = (
    spark.read
        .format("jdbc")
        .option("url", cc.create_jdbc())
        .option("driver", cc.get_Property("driver"))
        .option(
            "dbtable",
            "(select id, log_type, log_time from treasure_log) as subq"
        )
        .option("user", cc.get_Property("username"))
        .option("password", cc.get_Property("password"))
        .load()
        .filter(col("log_type") == 2)
)

if debugging_mode:
    print("Preview van date_src (extract):")
    date_src.show(5)

Preview van date_src (extract):
+--------------------+--------+--------------------+
|                  id|log_type|            log_time|
+--------------------+--------+--------------------+
|[00 00 0E 1D 1D 7...|       2|2021-07-24 19:40:...|
|[00 00 0E 57 98 D...|       2|2023-06-06 20:15:...|
|[00 00 18 C7 70 0...|       2|2021-08-04 09:46:...|
|[00 00 18 FC 4A 2...|       2|2023-06-29 20:38:...|
|[00 00 1B A5 89 B...|       2|2022-03-29 18:58:...|
+--------------------+--------+--------------------+
only showing top 5 rows


In [8]:
# TRANSFORM
neededDates = (
    date_src
        .withColumn("calendarDate", to_date(col("log_time")))
        .select("calendarDate")
        .distinct()
        .orderBy("calendarDate")
)
if debugging_mode:
    print("Unieke datums uit log_time (transform):")
    neededDates.show(10)

Unieke datums uit log_time (transform):
+------------+
|calendarDate|
+------------+
|  2020-09-11|
|  2020-09-12|
|  2020-09-13|
|  2020-09-14|
|  2020-09-15|
|  2020-09-16|
|  2020-09-17|
|  2020-09-18|
|  2020-09-19|
|  2020-09-20|
+------------+
only showing top 10 rows


In [9]:
# TRANSFORM
windowSpec = Window.orderBy("calendarDate")

dimDate = (
    neededDates
        .withColumn("DateSurKey", expr("uuid()"))
        .withColumn("DateId", row_number().over(windowSpec))
        .withColumn("Day", dayofmonth(col("calendarDate")))
        .withColumn("Week", weekofyear(col("calendarDate")))
        .withColumn("Month", date_format(col("calendarDate"), "MMMM"))
        .withColumn("Year", year(col("calendarDate")))
        .withColumn("MonthOfTheYear", month(col("calendarDate")))
        .withColumn("DayOfTheWeek", weekday(col("calendarDate")) + 1)  # 1 = maandag
        .withColumn("IsWeekDay", when(weekday(col("calendarDate")) < 5, True).otherwise(False))
        .select(
            "DateSurKey",
            "DateId",
            "Day",
            "Week",
            "Month",
            "Year",
            "MonthOfTheYear",
            "DayOfTheWeek",
            "IsWeekDay"
        )
)

if debugging_mode:
    print("DimDate (transform) preview:")
    dimDate.show(10)


DimDate (transform) preview:
+--------------------+------+---+----+---------+----+--------------+------------+---------+
|          DateSurKey|DateId|Day|Week|    Month|Year|MonthOfTheYear|DayOfTheWeek|IsWeekDay|
+--------------------+------+---+----+---------+----+--------------+------------+---------+
|f40af354-a01f-48d...|     1| 11|  37|September|2020|             9|           5|     true|
|48aa0ce5-ebe2-486...|     2| 12|  37|September|2020|             9|           6|    false|
|87d6fe2b-7d5a-4e0...|     3| 13|  37|September|2020|             9|           7|    false|
|74490479-8928-48c...|     4| 14|  38|September|2020|             9|           1|     true|
|19c867ba-1680-45c...|     5| 15|  38|September|2020|             9|           2|     true|
|e867067d-e64c-469...|     6| 16|  38|September|2020|             9|           3|     true|
|96a188ab-a604-4af...|     7| 17|  38|September|2020|             9|           4|     true|
|b9fc0f1a-493b-48c...|     8| 18|  38|September|202

In [10]:
#  LOAD
dimDate.write.format("delta").mode("overwrite").save("delta/DATE_DIM")

if debugging_mode:
    print("DimDate succesvol opgeslagen naar delta/DATE_DIM")


DimDate succesvol opgeslagen naar delta/DATE_DIM


In [11]:
spark.stop()